# Filtered 1D-Nanotube Template Forensics

Simple forensic analysis of the **filtered** pinning database
`nanotube_templates.npz`, built by `build_templates.py --filter` (quality gates in
`filter_templates.py`). A sibling of `comp_models/Analysis/nanotube_rtheta_forensics.ipynb`,
but it reads the compact CSR npz and reuses the exact metric/geometry helpers from
`filter_templates.py`, so the analysis and the DB filter always agree.

Every template here has already passed the 4 gates (contacts, hollow core, atom-count,
peaked ρ) — this notebook characterizes the survivors. **Writes nothing to disk.**

In [ ]:
# --- dependencies (run once) -------------------------------------------------
%pip install -q numpy pandas matplotlib scipy
import numpy, pandas, matplotlib, scipy
for m in (numpy, pandas, matplotlib, scipy):
    print(f"{m.__name__:12s} {m.__version__}")

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

HERE = Path.cwd()                     # run from NTGENS/data/nano_1D
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
import filter_templates as ft         # metrics, gates, geometry, draw_tube_3d

NPZ  = HERE / "nanotube_templates.npz"
SEED = 0
rng  = np.random.default_rng(SEED)

mpl.rcParams.update({
    "font.size": 12, "figure.dpi": 110, "figure.facecolor": "white",
    "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
    "axes.spines.right": False, "axes.titleweight": "bold",
})
ACCENT = ["#43AA8B", "#F8961E", "#F94144", "#277DA1"]
print("npz exists:", NPZ.exists(), "| thresholds:",
      dict(MIN_CONTACT=ft.MIN_CONTACT, MIN_RMIN=ft.MIN_RMIN,
           NATM=(ft.NATM_MIN, ft.NATM_MAX), PEAK_RATIO=ft.PEAK_RATIO))

## 1 · Load the filtered npz + compute metrics

In [ ]:
# Load the filtered CSR npz -> per-template records, and compute metrics for each.
templates = ft.load_npz_records(NPZ)
M = pd.DataFrame([dict(**ft.template_metrics(t["numbers"], t["frac"], t["cell"]),
                       source={0: "synthetic", 1: "real", -1: "untagged"}[t["source"]])
                  for t in templates])
print(f"Loaded {len(templates):,} filtered templates")
print("source split:", M.source.value_counts().to_dict())
display(M[["nsites", "r_min", "r_max", "min_nn", "peak_ratio"]].describe().round(2))

## 2 · Sanity check — all survivors clear the gates

In [ ]:
# Sanity: every template should already clear all 4 gates.
passes = np.array([ft.passes_filter(ft.template_metrics(t["numbers"], t["frac"], t["cell"]))
                   for t in templates])
print(f"pass all gates: {passes.sum():,} / {len(passes):,} "
      f"({100*passes.mean():.1f}%)  -> expect 100%")

## 3 · Population distributions

In [ ]:
# Population distributions of the gate metrics (survivors only).
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
specs = [("nsites", "atoms / cell", ft.NATM_MIN),
         ("r_min", "r$_{min}$ (Å)  [inner hollow radius]", ft.MIN_RMIN),
         ("r_max", "r$_{max}$ (Å)  [outer wall radius]", None),
         ("min_nn", "min contact (Å)", ft.MIN_CONTACT),
         ("peak_ratio", r"$\rho_{peak}/\bar{\rho}$", ft.PEAK_RATIO)]
for a, (col, lab, thr) in zip(ax.flat, specs):
    a.hist(M[col].dropna(), bins=40, color=ACCENT[3], edgecolor="white")
    if thr is not None:
        a.axvline(thr, color=ACCENT[2], ls="--", lw=2, label=f"gate = {thr}")
        a.legend()
    a.set_xlabel(lab); a.set_ylabel("templates")
# source split pie
a = ax.flat[5]; vc = M.source.value_counts()
a.pie(vc.values, labels=vc.index, autopct="%1.0f%%",
      colors=[ACCENT[0], ACCENT[1], ACCENT[2]][:len(vc)])
a.set_title("provenance"); a.grid(False)
fig.suptitle("Filtered template population — gate-metric distributions", fontweight="bold")
plt.tight_layout(); plt.show()

## 4 · Six random example templates

In [ ]:
# Six random filtered templates: metrics block + (θ, r) cross-section by element.
pick = rng.choice(len(templates), size=min(6, len(templates)), replace=False)
examples = [templates[i] for i in pick]
print("Six random filtered templates\n" + "=" * 60)
for t in examples:
    m = ft.template_metrics(t["numbers"], t["frac"], t["cell"])
    els = ft.symbols_of(t["numbers"]); formula = "".join(sorted(set(els)))
    print(f"\n─ {formula}  (N={m['nsites']}, {t['source'] if isinstance(t['source'],str) else ''})")
    print(f"   r_min={m['r_min']:.2f}  r_max={m['r_max']:.2f} Å   "
          f"min contact={m['min_nn']:.2f} Å   ρ_peak/ρ̄={m['peak_ratio']:.2f}")

fig, axes = plt.subplots(2, 3, figsize=(12, 8), subplot_kw=dict(projection="polar"))
for a, t in zip(axes.flat, examples):
    ax_i = ft.detect_tube_axis(t["frac"])
    r, th, z, u, v = ft.cylindrical_coords(t["cart"], t["cell"], ax_i)
    els = ft.symbols_of(t["numbers"]); uniq = sorted(set(els))
    cmap = {e: ft._OKABE[i % len(ft._OKABE)] for i, e in enumerate(uniq)}
    for e in uniq:
        mm = els == e
        a.scatter(th[mm], r[mm], s=20, color=cmap[e], label=e,
                  edgecolor="white", linewidth=0.3)
    a.set_title("".join(uniq) + f"  (N={t['nsites']})", fontsize=9, pad=8)
    a.set_xticklabels([]); a.set_yticklabels([])
    a.legend(fontsize=6, loc="upper right", bbox_to_anchor=(1.3, 1.15), framealpha=0.6)
fig.suptitle("Filtered templates — cross-section (θ, r) by element", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# The same 6 tubes in 3D perspective (r_min blue / r_max red cylinders, x/y/z Å).
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
fig = plt.figure(figsize=(15, 9.5))
for k, t in enumerate(examples):
    ax = fig.add_subplot(2, 3, k + 1, projection="3d")
    ft.draw_tube_3d(ax, t)
fig.suptitle("Filtered templates — 3D perspective with r$_{min}$/r$_{max}$ + x/y/z extents",
             y=0.99, fontweight="bold")
plt.tight_layout(); plt.show()